# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [1]:
ROLL_NO = 1024170343          # <-- your full roll number
NAME = "chhaya"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [2]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [3]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [4]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170343   Name: chhaya   Fingerprint: 7AB13307   Tickets: 15

Ticket_ID                                                                         Ticket_Text
      T01                                                Python 3.12 was installed yesterday.
      T02      I reinstalled the MATLAB while uploading the file. I have tried several times!
      T03                     The password keeps asking for login. I don't know what changed.
      T04                         I was locked out of the password on my laptop. Please help.
      T05                                The WiFi failed to open. I have tried several times!
      T06                 I was locked out of the password on my laptop. It worked yesterday.
      T07                          The server keeps disconnecting. Is there another solution?
      T08   Server keeps disconnecting when I use mobile hotspot. I have tried several times!
      T09                       The Python keeps disconnecting. It works on 

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [5]:
rows = []

for _, row in df.iterrows():
    result = process_ticket(row["Ticket_Text"])
    rows.append({
        "Ticket_ID": row["Ticket_ID"],
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation Tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

q1_df = pd.DataFrame(rows)
total_row = {
    "Ticket_ID": "TOTAL",
    "Sentences": q1_df["Sentences"].sum(),
    "Tokens": q1_df["Tokens"].sum(),
    "Punctuation Tokens": q1_df["Punctuation Tokens"].sum(),
    "Stopwords": q1_df["Stopwords"].sum(),
    "Unique Tokens": "—",
    "Final Tokens": q1_df["Final Tokens"].sum()
}
q1_df = pd.concat([q1_df, pd.DataFrame([total_row])], ignore_index=True)
display(q1_df)

focus_id = FOCUS[0]
focus_text = df.loc[df["Ticket_ID"] == focus_id, "Ticket_Text"].iloc[0]
focus_result = process_ticket(focus_text)
print("Selected ticket:", focus_id)
print("Raw text:", focus_text)
print("Actual tokens:", focus_result["tokens"])
print("Punctuation tokens:", focus_result["punct"])
print("Stopwords:", focus_result["stops"])
print("Final tokens:", focus_result["final"])

**Your answer:** Tokens such as `n't`, `'s`, `ca`, and `'m` can remain in the final tokens because `word_tokenize()` splits contractions into separate tokens. These fragments are not removed by the pipeline unless they are punctuation-only tokens or members of the English stopword list.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [6]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

import re

patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers": r"(?<![\w.])\d+(?:\.\d+)+(?![\w.])|(?<![\w.])\d+(?![\w.])",
    "Capitalized words": r"\b[A-Z][A-Za-z]*\b",
    "Words beginning with a vowel": r"\b[AEIOUaeiou][A-Za-z]*\b"
}

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_pattern = r"https?://[^\s<>]+|www\.[^\s<>]+"
phone_pattern = r"(?<!\d)(?:\+\d{1,3}[ .-]?)?(?:\d{5}[ .-]?\d{5}|\d{4}[ .-]?\d{3}[ .-]?\d{4})(?!\d)"

def extract_patterns(text):
    return {name: re.findall(pattern, text) for name, pattern in patterns.items()}

def replace_sensitive(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)
    return text

for _, row in df.iterrows():
    text = row["Ticket_Text"]
    print("\nTicket:", row["Ticket_ID"])
    for name, matches in extract_patterns(text).items():
        print(f"{name}: {matches}")
    print("Replaced:", replace_sensitive(text))

print("\nTEST_LINES")
for line in TEST_LINES:
    print("\nOriginal:", line)
    for name, matches in extract_patterns(line).items():
        print(f"{name}: {matches}")
    print("Replaced:", replace_sensitive(line))

from nltk.tokenize import sent_tokenize
capitalized_total = 0
sentence_initial_total = 0
for ticket_text in df["Ticket_Text"]:
    capitalized_total += len(re.findall(patterns["Capitalized words"], ticket_text))
    for sentence in sent_tokenize(ticket_text):
        words = re.findall(r"\b[A-Z][A-Za-z]*\b", sentence)
        if words and sentence.lstrip().startswith(words[0]):
            sentence_initial_total += 1
print("\nTotal capitalized-word matches:", capitalized_total)
print("Sentence-initial capitalized words (estimated, one per sentence):", sentence_initial_total)

**Your answer:** The preceding cell prints the total number of capitalized-word matches and an estimate of how many are sentence-initial. Sentence-initial words are often capitalized by grammar rather than because they are proper nouns, so this count helps distinguish sentence capitalization from meaningful capitalization.

---
## Q3. Custom tokenizer
Write `my_tokenize(text)` with one `RegexpTokenizer` pattern that keeps contractions (`isn't`), hyphenated words (`Wi-Fi`, `state-of-the-art`), decimals and versions (`95.5`, `2.0.1`), emails, URLs and phone numbers as single tokens. Run it on `TEST_LINES` and on your tickets, and compare with `word_tokenize`.

In [7]:
from nltk.tokenize import RegexpTokenizer

TOKEN_PATTERN = (
    r"https?://[^\s<>]+|www\.[^\s<>]+"
    r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    r"|(?:\+\d{1,3}[ .-]?)?(?:\d{5}[ .-]?\d{5}|\d{4}[ .-]?\d{3}[ .-]?\d{4})"
    r"|\d+(?:\.\d+)+%?"
    r"|\d+(?:,\d{3})+(?:\.\d+)?%?"
    r"|[A-Za-z]+(?:['’][A-Za-z]+)+"
    r"|[A-Za-z]+(?:-[A-Za-z]+)+"
    r"|[A-Za-z]+|\d+|[^\w\s]"
)

regexp_tokenizer = RegexpTokenizer(TOKEN_PATTERN)

def my_tokenize(text):
    return regexp_tokenizer.tokenize(text)

print("TEST_LINES comparison")
for line in TEST_LINES:
    print("\nText:", line)
    print("my_tokenize:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))

comparison_rows = []
for _, row in df.iterrows():
    custom_tokens = my_tokenize(row["Ticket_Text"])
    nltk_tokens = word_tokenize(row["Ticket_Text"])
    comparison_rows.append({
        "Ticket_ID": row["Ticket_ID"],
        "my_tokenize": custom_tokens,
        "word_tokenize": nltk_tokens,
        "Different": custom_tokens != nltk_tokens
    })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
different_tickets = int(comparison_df["Different"].sum())
print("Tickets tokenized differently:", different_tickets)

**Your answer:** The number of tickets tokenized differently is calculated and printed in the preceding cell as `different_tickets`. Differences occur because the custom tokenizer intentionally keeps contractions, hyphenated terms, URLs, emails, phone numbers, and version numbers together, while `word_tokenize()` may split them into multiple tokens.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [8]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

unique_final_tokens = sorted({
    token
    for _, row in df.iterrows()
    for token in process_ticket(row["Ticket_Text"])["final"]
    if any(ch.isalpha() for ch in token)
})

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer(r"(ing|ed|es|s)$", min=4)
lemmatizer = WordNetLemmatizer()

def wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

tagged_words = pos_tag(unique_final_tokens)
rows = []
for word, tag in tagged_words:
    p = porter.stem(word)
    l = lancaster.stem(word)
    r = regexp_stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, wordnet_pos(tag))
    if len({word, p, l, r, lemma}) > 1:
        rows.append({
            "Word": word,
            "POS": tag,
            "PorterStemmer": p,
            "LancasterStemmer": l,
            "RegexpStemmer": r,
            "WordNetLemmatizer": lemma
        })

q4_df = pd.DataFrame(rows)
display(q4_df)

**Your answer:** Over-stemming occurs when a stemmer removes too much and makes unrelated words share the same stem. Under-stemming occurs when related forms are not reduced to the same stem. Use the displayed comparison table to select examples from this generated dataset; the Lancaster stemmer is often more aggressive than Porter, while lemmatization generally returns dictionary forms based on part of speech.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [9]:
from collections import Counter
from nltk.util import bigrams
import matplotlib.pyplot as plt
import numpy as np

S1 = []
S2 = []
S3 = []
for _, row in df.iterrows():
    result = process_ticket(row["Ticket_Text"])
    S1.extend(result["tokens"])
    S2.extend(token for token in result["tokens"] if not is_punct(token))
    S3.extend(result["final"])
S4 = [porter.stem(token) for token in S3]

def print_stats(name, tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total else 0
    print(f"{name}: tokens={total}, unique tokens={unique}, TTR={ttr:.4f}")

print_stats("S1", S1)
print_stats("S2", S2)
print_stats("S3", S3)
print_stats("S4", S4)

print("\nTop 10 words in S1:")
print(Counter(S1).most_common(10))
print("\nTop 10 words in S3:")
print(Counter(S3).most_common(10))

frequencies = Counter(S3)
ranked_frequencies = sorted(frequencies.values(), reverse=True)
ranks = np.arange(1, len(ranked_frequencies) + 1)
plt.figure(figsize=(8, 5))
plt.plot(np.log(ranks), np.log(ranked_frequencies), marker="o", linestyle="none")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Zipf Plot for S3")
plt.grid(True)
plt.show()

print("\nTop 5 bigrams in S1:")
print(Counter(bigrams(S1)).most_common(5))
print("\nTop 5 bigrams in S3:")
print(Counter(bigrams(S3)).most_common(5))

**Your answer:** TTR is the number of unique tokens divided by the total number of tokens. From S1 to S4, punctuation and stopword removal reduce the token list, and stemming merges different word forms into shared stems. Therefore, both the numerator and denominator change, so TTR can increase or decrease depending on the generated tickets.